# Project: Medical Insurance Cost Prediction

#### 1. Import and Load the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
print("Shape of Data:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

#### 2. Exploratory Data Analysis (EDA)

In [ ]:
numerical_features = ['age', 'bmi', 'children', 'charges']
categorical_features = ['sex', 'smoker', 'region']

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_features):
    plt.subplot(2, 2, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
for i, col in enumerate(categorical_features):
    plt.subplot(1, 3, i+1)
    sns.countplot(x=col, data=df)
    plt.title(f'Count of {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='smoker', y='charges', data=df)
plt.title('Charges by Smoker Status')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df[numerical_features].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### EDA Insights
- **Smoker**: Strongest correlation with charges. Smokers pay significantly more.
- **Age**: Positive correlation with charges (older = higher cost).
- **BMI**: Skewed distribution. Some correlation with charges, likely higher for obese individuals.
- **Sex/Region**: Minimal impact observed in simple correlation.
- **Charges**: Right-skewed distribution.

#### 3. Missing Values & Outlier Treatment

In [ ]:
print(df.isnull().sum())

In [ ]:
# Outliers (Boxplots)
plt.figure(figsize=(15, 5))
for i, col in enumerate(['age', 'bmi', 'children']):
    plt.subplot(1, 3, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
plt.show()

#### 4. Feature Engineering & Preprocessing

In [ ]:
df_clean = df.copy()

# Encoding
le_sex = LabelEncoder()
le_smoker = LabelEncoder()

df_clean['sex'] = le_sex.fit_transform(df_clean['sex'])
df_clean['smoker'] = le_smoker.fit_transform(df_clean['smoker'])

# One-hot encode Region
df_clean = pd.get_dummies(df_clean, columns=['region'], drop_first=True)

# Split Features/Target
X = df_clean.drop('charges', axis=1)
y = df_clean['charges']

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()

cols_to_scale = ['age', 'bmi', 'children']
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

#### 5 & 6. Model Building & Evaluation

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    results.append({
        "Model": name,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "Train R2": train_r2,
        "Test R2": test_r2
    })

res_df = pd.DataFrame(results)
res_df

#### Overfitting Check
- If **Train R2 >> Test R2** (e.g., DT or RF having 0.99 train vs 0.85 test), the model is overfitting.

#### 7. Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5]
}

gb = GradientBoostingRegressor(random_state=42)
grid_search = GridSearchCV(estimator=gb, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

best_gb = grid_search.best_estimator_

print("Best Params:", grid_search.best_params_)
print("Best Grid R2:", grid_search.best_score_)

In [ ]:
# Evaluate Best Model
y_pred_tuned = best_gb.predict(X_test)
print("Tuned Test R2:", r2_score(y_test, y_pred_tuned))
print("Tuned Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_tuned)))

#### 8. Creation of a simple UI

In [ ]:
import os

# Create models dir
if not os.path.exists('../models'):
    os.makedirs('../models')

# Save artifacts
joblib.dump(best_gb, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_sex, '../models/le_sex.pkl')
joblib.dump(le_smoker, '../models/le_smoker.pkl')

print("Models saved to ../models/")

In [ ]:
import joblib
import pandas as pd

best_model = joblib.load('../models/best_model.pkl')
scaler = joblib.load('../models/scaler.pkl')
le_sex = joblib.load('../models/le_sex.pkl')
le_smoker = joblib.load('../models/le_smoker.pkl')

REGION_COLUMNS = ['region_northwest', 'region_southeast', 'region_southwest']

def predict_insurance_cost(age, sex, bmi, children, smoker, region):
    sample = pd.DataFrame({
        'age': [age],
        'sex': [sex.lower()],
        'bmi': [bmi],
        'children': [children],
        'smoker': [smoker.lower()],
    })
    sample['sex'] = le_sex.transform(sample['sex'])
    sample['smoker'] = le_smoker.transform(sample['smoker'])
    sample[['age','bmi','children']] = scaler.transform(sample[['age','bmi','children']])
    
    region_dummies = pd.DataFrame(0, index=[0], columns=REGION_COLUMNS)
    if region.lower() == 'northwest':
        region_dummies['region_northwest'] = 1
    elif region.lower() == 'southeast':
        region_dummies['region_southeast'] = 1
    elif region.lower() == 'southwest':
        region_dummies['region_southwest'] = 1
    sample = pd.concat([sample, region_dummies], axis=1)
    
    return best_model.predict(sample)[0]

# Interactive input system
if __name__ == "__main__":
    age = int(input("Enter age: "))
    sex = input("Enter sex (male/female): ")
    bmi = float(input("Enter BMI: "))
    children = int(input("Enter number of children: "))
    smoker = input("Smoker? (yes/no): ")
    region = input("Enter region (northwest/southeast/southwest/northeast): ")
    
    cost = predict_insurance_cost(age, sex, bmi, children, smoker, region)
    print(f"\nPredicted Insurance Cost: ₹{cost:.2f}")